# 05. Supplementary analysis

This notebook generates additional analyses used in the Supplementary Material.

The notebook includes:

- Supplementary Table S1: baseline clinical variables
- Supplementary Table S2: hyperparameter search space
- Supplementary Table S3: threshold sensitivity analysis
- Supplementary Table S4: robustness across random seeds
- Supplementary Table S5: extended comparison with prior studies
- Supplementary Table S6: feature ablation analysis
- Supplementary Table S7: permutation analysis
- Supplementary Figure S1: 
- Supplementary Figures S2-S4: latent-space and attention-based visualizations

## 1. Import libraries and setup

This section imports libraries used for supplementary analyses and creates output folders

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.manifold import TSNE

try:
    import umap
except ImportError:
    umap = None

SUPP_DIR = "../results/supplementary"
SUPP_FIGURE_DIR = "../results/supplementary/figures"

os.makedirs(SUPP_DIR, exist_ok=True)
os.makedirs(SUPP_FIGURE_DIR, exist_ok=True)

## 2. Supplementary Table S1: baseline clinical variables

The list of baseline clinical variables used in the primary ADNI model is provided in Supplementary Table S1.

This table includes variable categories, variable names, and brief descriptions for the 28 clinical predictors used in the main ADNI experiments.

## 3. Supplementary Table S2: hyperparameter search space and selected configuration

The hyperparameter search space and selected model configuration are provided in Supplementary Table S2.

Hyperparameter optimization was performed only on the pMCI vs. sMCI task, and the selected configuration was reused for the remaining binary classification tasks.

## 4. Supplementary Table S3: threshold sensitivity analysis

Threshold sensitivity is evaluated for the pMCI vs. sMCI task using the predicted probabilities from the trained model.

In [ ]:
task_name = "pMCI_vs_sMCI"

checkpoint, model, X_test, y_test, feature_cols = load_scaled_test_data(task_name)
y_prob = predict_probability(model, X_test)

threshold_results = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, zero_division=0)
    sen = recall_score(y_test, y_pred, zero_division=0)
    spe = tn / (tn + fp + 1e-12)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    threshold_results.append({
        "Threshold": threshold,
        "ACC": acc,
        "PRE": pre,
        "SEN": sen,
        "SPE": spe,
        "F1-score": f1,
    })

s3_df = pd.DataFrame(threshold_results)
s3_path = f"{SUPP_DIR}/supplementary_table_s3_threshold_sensitivity.csv"
s3_df.to_csv(s3_path, index=False)

s3_df

## 5. Supplementary Table S4: robustness across random seeds

The full model training and internal validation procedure is repeated across multiple random seeds.

In [ ]:
ROBUSTNESS_SEEDS = [17, 23, 25, 27, 37]

robustness_results = []

for seed in ROBUSTNESS_SEEDS:
    for task_name in TASKS:
        print(f"Seed {seed} | Task {task_name}")

        result, history_df, roc_info = train_and_evaluate_task(
            task_name=task_name,
            cfg=BEST_CFG,
            seed=seed,
        )

        robustness_results.append({
            "seed": seed,
            "task": task_name,
            "AUC": result["AUC"],
            "ACC": result["ACC"],
            "F1-score": result["F1"],
        })

robustness_df = pd.DataFrame(robustness_results)

s4_df = (
    robustness_df
    .groupby("task")
    .agg({
        "AUC": ["mean", "std"],
        "ACC": ["mean", "std"],
        "F1-score": ["mean", "std"],
    })
)

s4_df.columns = [
    "AUC_mean", "AUC_std",
    "ACC_mean", "ACC_std",
    "F1_mean", "F1_std",
]

s4_df = s4_df.reset_index()

s4_path = f"{SUPP_DIR}/supplementary_table_s4_seed_robustness.csv"
s4_df.to_csv(s4_path, index=False)

s4_df

## 6. Supplementary Figure S1: ROC curves

The ROC curves are generated in `02_model_training_and_internal_validation.ipynb`.

This section records the expected figure path for Supplementary Figure S1.

In [ ]:
s1_figure_path = "../results/figures/internal_roc_curves.png"

print("Supplementary Figure S1:")
print(s1_figure_path)

## 7. Supplementary Table S5: extended comparison with prior studies

The extended comparison with representative prior AD prediction studies is provided in Supplementary Table S5.

Because the values in this table are manually curated from previously published studies, this notebook does not regenerate the table computationally.

## 8. Supplementary Table S6: feature ablation analysis

For each feature, the model is retrained after removing the corresponding feature.          
The decrease in AUC relative to the baseline model is used as feature importance.

In [ ]:
def run_feature_ablation(task_name):
    checkpoint, model, X_test, y_test, feature_cols = load_scaled_test_data(task_name)

    baseline_auc = calculate_auc(model, X_test, y_test)
    rows = []

    for feature_to_remove in feature_cols:
        remaining_features = [
            feature for feature in feature_cols
            if feature != feature_to_remove
        ]

        result,_,_ = train_and_evaluate_task(
            task_name=task_name,
            cfg=BEST_CFG,
            seed=GLOBAL_SEED,
            feature_cols=remaining_features,
        )

        ablated_auc = result["AUC"]

        rows.append({
            "task": task_name,
            "removed_feature": feature_to_remove,
            "AUC": ablated_auc,
            "delta_AUC": baseline_auc - ablated_auc,
        })

    return pd.DataFrame(rows).sort_values("delta_AUC", ascending=False)

ablation_tasks = ["AD_vs_MCI", "MCI_vs_CN", "pMCI_vs_sMCI"]

ablation_df = pd.concat(
    [run_feature_ablation(task) for task in ablation_tasks],
    ignore_index=True,
)

s6_path = f"{SUPP_DIR}/supplementary_table_s6_feature_ablation.csv"
ablation_df.to_csv(s6_path, index=False)

ablation_df.groupby("task").head(5)

## 9. Supplementary Table S7: permutation analysis

Each feature is randomly permuted across test samples.  
The AUC decrease after permutation is used as feature importance.

In [ ]:
def run_permutation_analysis(task_name, seed=17, n_repeats=30):
    checkpoint, model, X_test, y_test, feature_cols = load_scaled_test_data(task_name)

    baseline_auc = calculate_auc(model, X_test, y_test)
    rows = []

    for j, feature in enumerate(feature_cols):
        permuted_aucs = []

        for repeat in range(n_repeats):
            rng = np.random.default_rng(seed + j * 10000 + repeat)

            X_perm = X_test.copy()
            X_perm[:, j] = rng.permutation(X_perm[:, j])

            permuted_auc = calculate_auc(model, X_perm, y_test)
            permuted_aucs.append(permuted_auc)

        mean_auc = np.mean(permuted_aucs)
        std_auc = np.std(permuted_aucs)

        rows.append({
            "task": task_name,
            "feature": feature,
            "baseline_AUC": baseline_auc,
            "AUC": mean_auc,
            "AUC_std": std_auc,
            "delta_AUC": baseline_auc - mean_auc,
        })

    return pd.DataFrame(rows).sort_values("delta_AUC", ascending=False)


permutation_df = pd.concat(
    [
        run_permutation_analysis(
            task,
            seed=GLOBAL_SEED,
            n_repeats=30,
        )
        for task in ablation_tasks
    ],
    ignore_index=True,
)

s7_path = f"{SUPP_DIR}/supplementary_table_s7_permutation.csv"
permutation_df.to_csv(s7_path, index=False)

permutation_df.groupby("task").head(5)

## 10. Supplementary Figures S2-S3: latent-space visualization

Latent representations are extracted from the trained model and visualized using t-SNE and UMAP.

In [ ]:
@torch.no_grad()
def extract_latent_representation(model, X):
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

    tokens = model.tokenizer(X_tensor)
    latent_tokens = model.sae_encoder(tokens)

    if model.use_cls:
        batch_size = X_tensor.size(0)
        cls_token = model.cls.expand(batch_size, 1, model.k)
        transformer_input = torch.cat([cls_token, latent_tokens], dim=1)
    else:
        transformer_input = latent_tokens

    contextual_tokens = model.transformer(
        transformer_input + model.pos[:, :transformer_input.size(1), :]
    )

    pooled = contextual_tokens[:, 0, :] if model.use_cls else contextual_tokens.mean(dim=1)

    return pooled.detach().cpu().numpy()

def plot_latent_space(task_name, method="tsne"):
    checkpoint, model, X_test, y_test, feature_cols = load_scaled_test_data(task_name)

    Z = extract_latent_representation(model, X_test)

    if method == "tsne":
        reducer = TSNE(n_components=2, random_state=GLOBAL_SEED, perplexity=min(30, len(y_test) - 1))
        Z_2d = reducer.fit_transform(Z)
    elif method == "umap":
        if umap is None:
            raise ImportError("Install umap-learn to generate UMAP visualization.")
        reducer = umap.UMAP(n_components=2, random_state=GLOBAL_SEED)
        Z_2d = reducer.fit_transform(Z)
    else:
        raise ValueError("method must be 'tsne' or 'umap'.")

    plt.figure(figsize=(5, 4))
    plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_test, s=20)
    plt.title(f"{task_name} - {method.upper()}")
    plt.xlabel(f"{method.upper()} 1")
    plt.ylabel(f"{method.upper()} 2")
    plt.tight_layout()

    output_path = f"{SUPP_FIGURE_DIR}/{method}_{task_name}.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return output_path

tsne_paths = []
umap_paths = []

for task_name in TASKS:
    tsne_paths.append(plot_latent_space(task_name, method="tsne"))

    if umap is not None:
        umap_paths.append(plot_latent_space(task_name, method="umap"))

print("t-SNE figures:", tsne_paths)
print("UMAP figures:", umap_paths)

## 11. Supplementary Figure S4: attention heatmaps

Attention heatmap generation depends on whether attention weights are explicitly stored during model forward passes.

If attention weights are not stored in the model, this section can be replaced with the attention extraction code used in the original experiment.

In [ ]:
print("Attention heatmap generation requires stored attention weights.")
print("If the trained model stores attention maps, load them here and save:")
print(f"{SUPP_FIGURE_DIR}/attention_heatmaps.png")

## 12. Expected outputs

After running this notebook, the following supplementary outputs should be generated locally:

```text
results/supplementary/supplementary_table_s1_variables.csv
results/supplementary/supplementary_table_s2_hyperparameters.csv
results/supplementary/supplementary_table_s3_threshold_sensitivity.csv
results/supplementary/supplementary_table_s4_seed_robustness.csv
results/supplementary/supplementary_table_s5_prior_study_comparison_template.csv
results/supplementary/supplementary_table_s6_feature_ablation.csv
results/supplementary/supplementary_table_s7_permutation.csv

results/supplementary/figures/tsne_AD_vs_CN.png
results/supplementary/figures/tsne_AD_vs_MCI.png
results/supplementary/figures/tsne_MCI_vs_CN.png
results/supplementary/figures/tsne_pMCI_vs_sMCI.png
results/supplementary/figures/umap_AD_vs_CN.png
results/supplementary/figures/umap_AD_vs_MCI.png
results/supplementary/figures/umap_MCI_vs_CN.png
results/supplementary/figures/umap_pMCI_vs_sMCI.png